In [1]:
import pandas as pd
import numpy as np
import re
import os
from datasets import Dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. 基础配置
model_path = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_path)
metric = evaluate.load("accuracy")

# 2. 数据加载与清洗 (延续之前的特征工程)
df = pd.read_csv('data/train.csv')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_df(input_df):
    temp_df = input_df.copy()
    temp_df['keyword'] = temp_df['keyword'].fillna('unknown')
    temp_df['text'] = temp_df['text'].apply(clean_text)
    # 拼接核心特征
    temp_df['text'] = "keyword: " + temp_df['keyword'].apply(clean_text) + ", text: " + temp_df['text']
    return temp_df.drop(['location', 'keyword'], axis=1, errors='ignore')

train_df_processed = preprocess_df(df)
dataset = Dataset.from_pandas(train_df_processed)

In [2]:
# 3. Tokenization (DeBERTa 建议 128 长度)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 4. 准备训练集和验证集
def transform_dataset(ds):
    ds = ds.remove_columns(["id", "text"])
    ds = ds.rename_column("target", "labels")
    return ds

final_dataset = transform_dataset(tokenized_dataset).train_test_split(test_size=0.1, seed=42)

Map:   0%|          | 0/7613 [00:00<?, ? examples/s]

In [3]:
# 5. 模型初始化与指标计算
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 修改第 6 部分的训练参数
training_args = TrainingArguments(
    output_dir="roberta_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,               # RoBERTa 的经典稳定学习率
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=16,
    num_train_epochs=3,               # 3 轮通常就能达到很好的效果
    weight_decay=0.01,                
    warmup_ratio=0.1,                 # 使用比例预热，更自然
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    bf16=True,                        # 重新开启 bf16，加速训练
    report_to="none",
    label_smoothing_factor=0.1,  # 告诉模型：不要太笃定，留 10% 的余地
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["test"],
    compute_metrics=compute_metrics,
)

# 7. 开始训练
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.467953,0.833333
2,0.516321,0.456688,0.842520
3,0.411886,0.498080,0.824147


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1287, training_loss=0.4473533126487228, metrics={'train_runtime': 66.2471, 'train_samples_per_second': 310.248, 'train_steps_per_second': 19.427, 'total_flos': 1351930380203520.0, 'train_loss': 0.4473533126487228, 'epoch': 3.0})

In [4]:
# 8. 测试集预测与提交
test_df = pd.read_csv('data/test.csv')
test_ids = test_df['id']
test_df_processed = preprocess_df(test_df)
test_dataset = Dataset.from_pandas(test_df_processed)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)

submission = pd.DataFrame({'id': test_ids, 'target': preds})
os.makedirs('outputs', exist_ok=True)
submission.to_csv('outputs/submission_roberta.csv', index=False)
print("roBERTa Submission saved successfully!")

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

roBERTa Submission saved successfully!


# 尝试5-fold交叉验证

In [5]:
import torch
import gc
from sklearn.model_selection import StratifiedKFold
from scipy.special import softmax

# ================= 1. 基础配置 =================
MODEL_PATH = "roberta-base"
N_FOLDS = 5
MAX_LEN = 128
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
metric = evaluate.load("accuracy")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_df(input_df, is_test=False):
    temp_df = input_df.copy()
    temp_df['keyword'] = temp_df['keyword'].fillna('unknown')
    temp_df['text'] = temp_df['text'].apply(clean_text)
    temp_df['text'] = "keyword: " + temp_df['keyword'].apply(clean_text) + ", text: " + temp_df['text']
    
    drop_cols = ['location', 'keyword']
    if not is_test and 'id' in temp_df.columns:
        drop_cols.append('id')
    return temp_df.drop(drop_cols, axis=1, errors='ignore')

In [6]:
# ================= 2. 准备数据 =================
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

train_df_processed = preprocess_df(train_df)
test_df_processed = preprocess_df(test_df, is_test=True)

test_ids = test_df_processed['id']
test_df_processed = test_df_processed.drop(['id'], axis=1)

# 测试集 Tokenization
test_dataset = Dataset.from_pandas(test_df_processed)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

In [7]:
# ================= 3. 5-Fold 交叉验证核心循环 =================
# 初始化一个数组，用来累加 5 个模型对测试集的预测概率
test_preds_total = np.zeros((len(test_df), 2)) 

# 使用分层 K 折，保证正负样本比例一致
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(train_df_processed, train_df_processed['target'])):
    print(f"========== 🚀 开始训练 Fold {fold + 1} / {N_FOLDS} ==========")
    
    # 划分当前 Fold 的训练集和验证集
    fold_train_df = train_df_processed.iloc[train_idx].rename(columns={"target": "labels"})
    fold_valid_df = train_df_processed.iloc[valid_idx].rename(columns={"target": "labels"})
    
    train_ds = Dataset.from_pandas(fold_train_df).map(tokenize_function, batched=True)
    valid_ds = Dataset.from_pandas(fold_valid_df).map(tokenize_function, batched=True)
    
    # ⚠️ 极其关键：每次循环必须重新初始化模型，否则会拿上一折训练好的权重继续练！
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=2)
    
    training_args = TrainingArguments(
        output_dir=f"roberta_output_fold_{fold+1}", # 每一折的输出分文件夹存放
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_ratio=0.1,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        label_smoothing_factor=0.1,  # 你刚才加上的上分利器
        bf16=True,                   # 充分利用你的高级显卡
        report_to="none",
        save_total_limit=1           # 限制保存的 checkpoint 数量，省点硬盘空间
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics,
    )
    
    # 开始训练当前折
    trainer.train()
    
    # 当前折训练完毕，立刻对 Kaggle 测试集进行预测
    print(f"--- 正在使用 Fold {fold + 1} 的最佳模型预测测试集 ---")
    predictions = trainer.predict(tokenized_test)
    
    # 将模型输出的 logits 转换为 0~1 的概率，并累加到总数组中
    fold_probs = softmax(predictions.predictions, axis=1)
    test_preds_total += fold_probs / N_FOLDS
    
    # 释放显存，防止 OOM (Out Of Memory)
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

print("========== 🎉 所有 Fold 训练完毕！ ==========")

========== 🚀 开始训练 Fold 1 / 5 ==========


Map:   0%|          | 0/6090 [00:00<?, ? examples/s]

Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--- 正在使用 Fold 5 的最佳模型预测测试集 ---


========== 🎉 所有 Fold 训练完毕！ ==========


In [8]:
# ================= 4. 生成最终融合结果 =================
# test_preds_total 目前是 5 个模型预测概率的平均值
# 直接取概率最大的一列作为最终预测
final_preds = np.argmax(test_preds_total, axis=1)

submission = pd.DataFrame({'id': test_ids, 'target': final_preds})
os.makedirs('outputs', exist_ok=True)
submission.to_csv('outputs/submission_roberta_5fold_smooth.csv', index=False)
print("roBERTa 5fold Submission saved successfully!")

roBERTa 5fold Submission saved successfully!
